# 06 — Draw Probability Diagnostic (Stage A)

Is the "0 draws predicted" a real probability failure, or just an `argmax` artifact?

Reads the walk-forward backtest output `reports/predictions.parquet`. The key number is the
**draw-class one-vs-rest AUC**: ~0.50 means `p_draw` cannot rank draw matches above non-draws
(no draw signal in the features); well above 0.50 would mean the probabilities are healthy and
only the hard `argmax` decision is the problem.

**Decision gate** — if `p_draw` sits near the base rate but can't separate draws from non-draws
(AUC ~0.50, 2nd-vs-last rank a coin toss): magnitude is fine, *discrimination* is absent, and the
real lever is richer inputs (xG / more seasons), not re-tuning.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# Make the project root importable (this notebook lives in notebooks/)
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

# Shared conventions (match notebooks 03/04/05)
LABEL_ORDER = ["HW", "D", "AW"]
PALETTE = {"HW": "#2a9d8f", "D": "#e9c46a", "AW": "#e76f51"}
PROB_COLS = ["p_home", "p_draw", "p_away"]

preds = pd.read_parquet(ROOT / "reports" / "predictions.parquet")
print("models:", preds["model"].unique().tolist())
print("rows per model:", len(preds[preds["model"] == "rf_tuned"]))

## Draw probability for the main model (`rf_tuned`)

In [ ]:
MODEL = "rf_tuned"
d = preds[preds["model"] == MODEL]
P = d[PROB_COLS].to_numpy()
is_draw = (d["y_true"].to_numpy() == "D")

# Magnitude vs signal: is p_draw reasonable in size, and is it ever the modal class?
print(f"{MODEL}: draws in test = {is_draw.sum()} / {len(d)} ({is_draw.mean():.1%})")
print(f"mean p_draw = {P[:,1].mean():.3f}  (base rate {is_draw.mean():.3f})   max p_draw = {P[:,1].max():.3f}")
print(f"times draw is the argmax = {(P.argmax(1) == 1).sum()}")
# Rank of the draw class per match: 1st (argmax) / 2nd / 3rd. A ~50/50 2nd-vs-last split = chance.
draw_rank = (-P).argsort(1).argsort(1)[:, 1]  # 0 = highest, 2 = lowest
print(f"draw ranked 1st = {(draw_rank == 0).sum()}, 2nd = {(draw_rank == 1).sum()}, last = {(draw_rank == 2).sum()}")
print(f"mean p_draw on real draws = {P[is_draw,1].mean():.3f}  vs non-draws = {P[~is_draw,1].mean():.3f}")

# If these two distributions sit on top of each other, p_draw carries no draw signal.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(P[is_draw, 1], bins=25, alpha=0.6, density=True, color=PALETTE["D"], label="actual draws")
ax.hist(P[~is_draw, 1], bins=25, alpha=0.6, density=True, color="#888888", label="actual non-draws")
ax.axvline(is_draw.mean(), color="black", ls="--", lw=1, label="base rate")
ax.set_xlabel("p_draw"); ax.set_ylabel("density")
ax.set_title(f"p_draw — draws vs non-draws ({MODEL})")
ax.legend(); fig.tight_layout(); plt.show()

## Draw discrimination across every model

In [ ]:
# AUC ~0.50 = cannot rank draws above non-draws. argmax_draws = how many draws the model ever picks.
rows = []
for m, g in preds.groupby("model", sort=False):
    yt = (g["y_true"].to_numpy() == "D").astype(int)
    pdr = g["p_draw"].to_numpy()
    Pm = g[PROB_COLS].to_numpy()
    rows.append({
        "model": m,
        "draw_AUC": round(roc_auc_score(yt, pdr), 3),
        "mean_p_draw": round(pdr.mean(), 3),
        "argmax_draws": int((Pm.argmax(1) == 1).sum()),
    })
draw_diag = pd.DataFrame(rows)
print("actual draw base rate:", round((preds[preds.model == "rf_tuned"]["y_true"] == "D").mean(), 3))
draw_diag